Thiago Dellano Machado de Oliveira | 24101272

# Acesso a Banco de Dados com JDBC

## Introdução aos Bancos de Dados Relacionais

Um **banco de dados relacional (RDBMS)** organiza dados em **tabelas** (linhas e colunas), com relações entre elas. É amplamente utilizado por garantir consistência, integridade e permitir consultas complexas com SQL.

1. **Organização dos dados**: Os dados são armazenados em **tabelas**, compostas por linhas (registros) e colunas (atributos).
2. **Chaves primárias e estrangeiras**:

   * **Chave primária**: Identificador único de cada registro em uma tabela.
   * **Chave estrangeira**: Estabelece uma ligação entre duas tabelas.
3. **Comparação com bancos orientados a documentos**:

   * **Relacional**: Estrutura tabular, forte consistência, ideal para dados estruturados.
   * **Documental (NoSQL)**: Estrutura flexível (JSON/XML), ideal para dados semi-estruturados, escalabilidade horizontal.

---

## Banco de Dados – Exemplo: Livros

1. **Tabelas e colunas**:

   * `livros(id, titulo, autor_id, ano_publicacao)`
   * `autores(id, nome)`
2. **Relacionamento**:

   * `livros.autor_id` é chave estrangeira referenciando `autores.id`.
3. **Integridade Referencial**:

   * Garante que valores da chave estrangeira existam na tabela referenciada.
   * Evita registros órfãos e mantém a consistência.

---

## Instruções SQL Básicas

```sql
-- 1. SELECT com colunas específicas
SELECT titulo, ano_publicacao FROM livros;

-- 2. WHERE para filtrar
SELECT * FROM livros WHERE ano_publicacao > 2020;

-- 3. ORDER BY
SELECT * FROM livros ORDER BY ano_publicacao DESC;

-- 4. JOIN entre livros e autores
SELECT livros.titulo, autores.nome 
FROM livros 
JOIN autores ON livros.autor_id = autores.id;

-- 5. INSERT
INSERT INTO livros (titulo, autor_id, ano_publicacao) VALUES ('Livro X', 1, 2023);

-- 6. UPDATE
UPDATE livros SET ano_publicacao = 2024 WHERE id = 1;

-- 7. DELETE
DELETE FROM livros WHERE id = 2;
```

---

## Conexão com o Banco via JDBC

```java
import java.sql.*;

public class ConexaoDB {
    public static void main(String[] args) {
        String url = "jdbc:mysql://localhost:3306/biblioteca";
        String usuario = "root";
        String senha = "1234";

        try (Connection conn = DriverManager.getConnection(url, usuario, senha)) {
            System.out.println("Conectado!");
        } catch (SQLException e) {
            e.printStackTrace();
        }
    }
}
```

1. **Driver JDBC**: A detecção é automática via `Class.forName()` em versões antigas ou via `META-INF/services/java.sql.Driver`.
2. **Criar Statement**:

   ```java
   Statement stmt = conn.createStatement();
   ResultSet rs = stmt.executeQuery("SELECT * FROM livros");
   ```

---

## Leitura de Dados com ResultSet

1. **Iterar sobre ResultSet**:

   ```java
   while (rs.next()) {
       System.out.println(rs.getString("titulo"));
   }
   ```
2. **Recuperar dados por nome de coluna**:

   * `rs.getInt("id")`
   * `rs.getString("titulo")`
3. **ResultSetMetaData**:

   ```java
   ResultSetMetaData meta = rs.getMetaData();
   int colunas = meta.getColumnCount();
   for (int i = 1; i <= colunas; i++) {
       System.out.println(meta.getColumnName(i));
   }
   ```

---

## Populando JTable com Dados

1. **Classe `ResultSetTableModel`**:

   ```java
   public class ResultSetTableModel extends AbstractTableModel {
       private ResultSet rs;
       private ResultSetMetaData rsmd;

       public ResultSetTableModel(ResultSet rs) throws SQLException {
           this.rs = rs;
           this.rsmd = rs.getMetaData();
       }

       // Implementações dos métodos getColumnCount, getRowCount, getValueAt
   }
   ```

2. **Ordenação e filtro com RowFilter**:

   ```java
   TableRowSorter<TableModel> sorter = new TableRowSorter<>(tabela.getModel());
   tabela.setRowSorter(sorter);
   sorter.setRowFilter(RowFilter.regexFilter("busca"));
   ```

---

## RowSet e Conexões Simplificadas

1. **Diferenças**:

   * `JdbcRowSet`: Conectado, baseado em ResultSet.
   * `CachedRowSet`: Desconectado, serializável.
2. **Configuração de JdbcRowSet**:

   ```java
   JdbcRowSet rowSet = RowSetProvider.newFactory().createJdbcRowSet();
   rowSet.setUrl("jdbc:mysql://localhost:3306/biblioteca");
   rowSet.setUsername("root");
   rowSet.setPassword("1234");
   rowSet.setCommand("SELECT * FROM livros");
   rowSet.execute();
   ```
3. **Serialização de CachedRowSet**:

   * Permite enviar dados desconectados via rede ou gravar em arquivos.

---

## Uso de PreparedStatements

1. **Consulta parametrizada com INSERT**:

   ```java
   PreparedStatement ps = conn.prepareStatement("INSERT INTO livros (titulo, autor_id, ano_publicacao) VALUES (?, ?, ?)");
   ps.setString(1, "Novo Livro");
   ps.setInt(2, 1);
   ps.setInt(3, 2024);
   ps.executeUpdate();
   ```
2. **Setar parâmetros**: Usando métodos como `setString`, `setInt`, etc.
3. **LIKE com parâmetros**:

   ```java
   PreparedStatement ps = conn.prepareStatement("SELECT * FROM livros WHERE titulo LIKE ?");
   ps.setString(1, "%Java%");
   ```

---

## Procedures Armazenadas

1. **`CallableStatement`**: Usado para chamar procedures armazenadas no banco.
2. **Exemplo com entrada e saída**:

   ```java
   CallableStatement cs = conn.prepareCall("{call calcular_estatistica(?, ?)}");
   cs.setInt(1, 10); // parâmetro de entrada
   cs.registerOutParameter(2, Types.INTEGER); // parâmetro de saída
   cs.execute();
   int resultado = cs.getInt(2);
   ```

---

## Processamento de Transações

1. **Transações e propriedades ACID**:

   * **Atomicidade**: Tudo ou nada.
   * **Consistência**: Banco permanece íntegro.
   * **Isolamento**: Execuções paralelas não interferem.
   * **Durabilidade**: Alterações persistem após commit.

2. **Exemplo com commit e rollback**:

   ```java
   conn.setAutoCommit(false);
   try {
       PreparedStatement ps1 = conn.prepareStatement(...);
       ps1.executeUpdate();

       PreparedStatement ps2 = conn.prepareStatement(...);
       ps2.executeUpdate();

       conn.commit(); // Se tudo der certo
   } catch (SQLException e) {
       conn.rollback(); // Reverte alterações em caso de erro
   }
   ```